# Watermark Teks "sabila" pada Domain Pixel — Diuji terhadap Kompresi Neural Codec

```
Dataset -> Video -> Frame -> payload 48 bit "sabila" -> embedding pixel-domain -> frame ber-watermark
   -> Neural Codec compress() -> compressed representation (bitstream) -> Neural Codec decompress()
   -> frame hasil rekonstruksi -> ekstraksi pixel-domain -> payload 48 bit
   -> BER / Bit Accuracy / Deteksi / PSNR / BPP
```

- Payload `"sabila"` = 6 byte = **48 bit**.
- Watermark ditanam **langsung pada nilai pixel**. Tidak ada neural watermark encoder/decoder, tidak ada training.
- Neural Codec (`bmshj2018-hyperprior`, CompressAI, pretrained, dibekukan) hanya dipakai sebagai **penguji kompresi/rekonstruksi**.
- Ekstraksi dilakukan pada **frame hasil rekonstruksi**, bukan pada frame asli/ber-watermark.
- Tidak ada ECC dan tidak ada perbandingan multi-codec pada versi ini.
- Bukti eksperimen ada di bagian **Visual / Audit** (bukti Neural Codec, mapping payload, audit pixel, bukti ekstraksi, strength sweep untuk imperceptibility). Cell terakhir = ringkasan visual 4 panel.
- Semua path berasal dari satu `ROOT` di Shared Drive; setiap run punya `run_id`, checkpoint atomic, dan resume fail-fast.

In [ ]:
import sys, subprocess, importlib.util, shutil

def pip(*pkgs, no_deps=False):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + (['--no-deps'] if no_deps else []) + list(pkgs), check=False)

# --no-deps: metadata compressai meminta numpy<2, sedangkan Colab memakai numpy 2.x
if importlib.util.find_spec('compressai') is None:
    pip('compressai==1.2.8', 'pytorch-msssim', 'einops', no_deps=True)
if importlib.util.find_spec('torch_geometric') is None:   # diimpor compressai.__init__
    pip('torch_geometric')
if shutil.which('ffmpeg') is None or shutil.which('ffprobe') is None:
    subprocess.run('apt-get -y install ffmpeg > /dev/null 2>&1', shell=True)
assert shutil.which('ffmpeg') and shutil.which('ffprobe'), 'ffmpeg / ffprobe tidak tersedia'

if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')

In [ ]:
import os
import json
import random
import hashlib
import subprocess
import tempfile
import shutil
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display, HTML

from tqdm.auto import tqdm
from scipy.stats import binom
from scipy.ndimage import gaussian_filter

## Konfigurasi, Run ID, dan sistem checkpoint

- Satu root: `ROOT`. Dataset **dibaca langsung** dari `ROOT/test` (tidak disalin ke `/content`).
- `run_id` = timestamp + hash pendek konfigurasi, dan **tidak boleh dipakai ulang untuk konfigurasi lain**.
- Checkpoint selalu = stage terakhir yang **sudah dipersist**: proses -> tulis artifact -> verifikasi artifact -> update checkpoint.
- Saat resume: JSON checkpoint, `run_id`, konfigurasi, dan artifact (ada + SHA-256 cocok + bisa dibaca) divalidasi. Gagal = **berhenti dengan error**, tidak diam-diam mengulang/menimpa.

In [ ]:
ROOT = Path('/content/drive/MyDrive/order_20260827_213103')
DATASET_DIR = ROOT / 'test'
OUT_ROOT = ROOT / 'output'
FRAMES_ROOT = ROOT / 'frames_tmp'
CKPT_ROOT = ROOT / 'checkpoints'

# ---- konfigurasi utama ----
PAYLOAD_TEXT = 'sabila'
FRAME_SIZE = (192, 256)            # (H, W), wajib kelipatan 64 untuk bmshj2018-hyperprior
GRID, CELL = (6, 8), 8             # 6x8 = 48 blok (1 blok = 1 bit); pola ±1 berukuran sel 8x8 pixel
STRENGTH = 6
KEY = 2024
CODEC_NAME = 'bmshj2018-hyperprior'
CODEC_QUALITY = 3
CODEC_BATCH = 16
P_FALSE_ALARM = 1e-4
SEED = 42
MAX_FRAMES, MAX_VIDEOS = 100, 30
QUICK_TEST = False                 # True = uji alur cepat (hasil tidak representatif)
RESUME_RUN_ID = None               # isi dengan run_id lama untuk melanjutkan run tersebut
if QUICK_TEST:
    MAX_FRAMES, MAX_VIDEOS = 16, 2

# ---- audit & imperceptibility (tidak masuk CONFIG/run_id: hanya mengatur tampilan dan strength sweep) ----
AUDIT_VIDEO_INDEX = 0             # video yang diaudit & ditampilkan di ringkasan akhir (indeks pada daftar video terurut)
AUDIT_FRAME_IDX = 0               # frame yang diaudit & ditampilkan
AUDIT_BLOCK = 0                   # blok (0..47) yang ditampilkan lengkap nilai pixel-nya
SWEEP_STRENGTHS = [1, 2, 3, 4, 5, 6]
SWEEP_MAX_VIDEOS = 10             # jumlah video (tersebar merata) untuk strength sweep

PAYLOAD_BITS_STR = ''.join(f'{b:08b}' for b in PAYLOAD_TEXT.encode())
EMBED_CFG = {'payload': PAYLOAD_TEXT, 'grid': list(GRID), 'cell': CELL, 'strength': STRENGTH, 'key': KEY,
             'frame_size': list(FRAME_SIZE), 'max_frames': MAX_FRAMES}
CODEC_CFG = {'codec': CODEC_NAME, 'quality': CODEC_QUALITY, 'pretrained': True, 'frozen': True, 'batch': CODEC_BATCH}
CONFIG = {'embedding': EMBED_CFG, 'codec': CODEC_CFG, 'p_false_alarm': P_FALSE_ALARM,
          'max_videos': MAX_VIDEOS, 'seed': SEED}

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

assert ROOT.is_dir(), f'ROOT tidak ditemukan: {ROOT}'
assert DATASET_DIR.is_dir(), f'dataset tidak ditemukan: {DATASET_DIR}'

STAGES = ['INITIALIZED', 'DATASET_READY', 'FRAMES_EXTRACTED', 'WATERMARK_EMBEDDED', 'CODEC_COMPRESSED',
          'CODEC_RECONSTRUCTED', 'WATERMARK_EXTRACTED', 'EVALUATED', 'COMPLETED']


class CheckpointError(RuntimeError):
    pass


def utc_now():
    return datetime.now(timezone.utc).isoformat(timespec='seconds')


def file_sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


# ---- penulisan atomic: tulis ke *.tmp, flush+fsync, lalu os.replace ----
def _atomic_write(path, writer):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.tmp')
    with open(tmp, 'wb') as f:
        writer(f); f.flush(); os.fsync(f.fileno())
    os.replace(tmp, path)


def atomic_write_json(path, obj):
    _atomic_write(path, lambda f: f.write(json.dumps(obj, indent=2, ensure_ascii=False).encode('utf-8')))


def atomic_save_npy(path, arr):
    _atomic_write(path, lambda f: np.save(f, arr))


def atomic_copy(src, dst):
    _atomic_write(dst, lambda f: f.write(Path(src).read_bytes()))


def make_artifact(path, shape=None):
    '''Deskripsi artifact yang sudah ditulis: path + SHA-256 (+ shape untuk .npy).'''
    path = Path(path)
    return {'path': str(path), 'sha256': file_sha256(path), 'shape': list(shape) if shape is not None else None}


def verify_entry(entry):
    '''Validasi artifact sebuah entry checkpoint. Hilang / kosong / corrupt / tidak terbaca -> CheckpointError.'''
    try:
        for label, a in entry['artifacts'].items():
            p = Path(a['path'])
            if not p.is_file() or p.stat().st_size == 0:
                raise CheckpointError(f'artifact hilang/kosong ({label}): {p}')
            if file_sha256(p) != a['sha256']:
                raise CheckpointError(f'artifact corrupt, SHA-256 tidak cocok ({label}): {p}')
            if p.suffix == '.npy':
                try:
                    arr = np.load(p, mmap_mode='r')
                except Exception as e:
                    raise CheckpointError(f'artifact tidak dapat dibaca ({label}): {p}: {e}')
                if arr.dtype != np.uint8 or list(arr.shape) != a['shape']:
                    raise CheckpointError(f'artifact {label} dtype/shape tidak sesuai checkpoint: {p}')
            elif p.suffix == '.json':
                try:
                    json.loads(p.read_text(encoding='utf-8'))
                except Exception as e:
                    raise CheckpointError(f'artifact JSON tidak dapat dibaca ({label}): {p}: {e}')
    except KeyError as e:
        raise CheckpointError(f'entry checkpoint tidak lengkap, field hilang: {e}')
    return entry


def load_npy(entry, label):
    return np.load(entry['artifacts'][label]['path'])


class Checkpoint:
    '''Satu shard checkpoint (JSON). Selalu = stage terakhir yang berhasil dipersist.'''

    def __init__(self, name):
        self.name = name
        self.path = CKPT_DIR / f'{name}.json'
        self.state = None

    def _fresh(self):
        return {'run_id': RUN_ID, 'config_hash': CONFIG_HASH, 'pipeline_stage': 'INITIALIZED',
                'video_index': None, 'frame_index': None, 'payload': PAYLOAD_TEXT, 'payload_bits': PAYLOAD_BITS_STR,
                'embedding_configuration': EMBED_CFG, 'codec_configuration': CODEC_CFG,
                'output_paths': {}, 'status': 'INITIALIZED', 'timestamp': utc_now(), 'done': {}}

    def load(self):
        if not self.path.exists():
            self.state = self._fresh()          # shard belum pernah ditulis = stage ini belum dimulai
            return self
        try:
            st = json.loads(self.path.read_text(encoding='utf-8'))
        except Exception as e:
            raise CheckpointError(f'checkpoint corrupt / tidak dapat dibaca: {self.path}: {e}')
        if not isinstance(st, dict) or not isinstance(st.get('done'), dict) or st.get('pipeline_stage') not in STAGES:
            raise CheckpointError(f'struktur checkpoint tidak valid: {self.path}')
        for k, want in [('run_id', RUN_ID), ('config_hash', CONFIG_HASH), ('payload_bits', PAYLOAD_BITS_STR),
                        ('embedding_configuration', EMBED_CFG), ('codec_configuration', CODEC_CFG)]:
            if st.get(k) != want:
                raise CheckpointError(f'checkpoint {self.path} tidak cocok pada "{k}": {st.get(k)!r} != {want!r}')
        self.state = st
        return self

    def commit(self, stage=None, status=None, video=None, entry=None, video_index=None, frame_index=None):
        st = self.state
        if video is not None:
            st['done'][video] = entry
            st['output_paths'][video] = {k: a['path'] for k, a in entry['artifacts'].items()}
        if video_index is not None:
            st['video_index'] = video_index
        if frame_index is not None:
            st['frame_index'] = frame_index
        if stage and STAGES.index(stage) >= STAGES.index(st['pipeline_stage']):
            st['pipeline_stage'] = stage
        if status:
            st['status'] = status
        st['timestamp'] = utc_now()
        atomic_write_json(self.path, st)


# ---- run_id: timestamp + hash pendek konfigurasi ----
CONFIG_HASH = hashlib.sha256(json.dumps(CONFIG, sort_keys=True).encode()).hexdigest()[:6]
if RESUME_RUN_ID:
    RUN_ID = RESUME_RUN_ID
    if not RUN_ID.endswith('_' + CONFIG_HASH):
        raise CheckpointError(f'run_id {RUN_ID} bukan milik konfigurasi ini (hash konfigurasi sekarang: {CONFIG_HASH})')
else:
    RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S') + '_' + CONFIG_HASH

RUN_DIR = OUT_ROOT / RUN_ID
DIRS = {k: RUN_DIR / k for k in ['metrics', 'watermarked', 'compressed', 'reconstructed', 'reports']}
FRAMES_DIR = FRAMES_ROOT / RUN_ID
CKPT_DIR = CKPT_ROOT / RUN_ID

if RESUME_RUN_ID:
    if not RUN_DIR.is_dir():
        raise CheckpointError(f'RESUME_RUN_ID={RUN_ID}, tetapi {RUN_DIR} tidak ada')
else:
    if RUN_DIR.exists() or CKPT_DIR.exists() or FRAMES_DIR.exists():
        raise CheckpointError(f'run_id {RUN_ID} sudah dipakai; tidak boleh menimpa run lain')
for d in [*DIRS.values(), FRAMES_DIR, CKPT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

ckpt_run = Checkpoint('run').load()
ckpt_run.commit(stage='INITIALIZED', status='RESUMED' if RESUME_RUN_ID else 'STARTED')
print('RUN_ID    :', RUN_ID)
print('RUN_DIR   :', RUN_DIR)
print('Checkpoint:', CKPT_DIR)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Neural Codec (CompressAI, pretrained, dibekukan)

`codec.compress()` menghasilkan `strings` (**compressed representation / bitstream**) dan `shape`. `codec.decompress()` merekonstruksi frame.
`strings` **bukan** latent. BPP dihitung dari **seluruh byte** di `strings` (semua bagian, semua batch), dibagi `T*H*W` frame yang dikompresi.

In [ ]:
from compressai.zoo import image_models

codec = image_models[CODEC_NAME](quality=CODEC_QUALITY, pretrained=True).to(device).eval()
for p in codec.parameters():
    p.requires_grad_(False)                       # frozen: tanpa training, tanpa optimizer
codec.update(force=True)


def to_tensor(f):
    '''uint8 (N,H,W,3) RGB -> float32 (N,3,H,W) di [0,1].'''
    assert f.dtype == np.uint8 and f.ndim == 4 and f.shape[-1] == 3, f'frame harus uint8 (N,H,W,3), dapat {f.dtype} {f.shape}'
    return torch.from_numpy(np.ascontiguousarray(f)).permute(0, 3, 1, 2).float().div(255.0)


def to_uint8(x):
    '''float (N,3,H,W) -> uint8 (N,H,W,3), di-clip ke [0,1].'''
    return x.clamp(0, 1).mul(255.0).round().byte().permute(0, 2, 3, 1).cpu().numpy()


def compressed_bytes(strings):
    '''Total byte seluruh compressed representation: strings = list bagian (y, z), tiap bagian list bytes per item batch.'''
    return int(sum(len(s) for part in strings for s in part))


@torch.no_grad()
def codec_roundtrip(frames, batch=CODEC_BATCH):
    '''frame uint8 (T,H,W,3) -> compress() -> strings+shape -> decompress() -> frame rekonstruksi uint8.
    Return (rekonstruksi, meta). meta = metadata audit compressed representation (bukan latent).'''
    T, H, W, _ = frames.shape
    assert H % 64 == 0 and W % 64 == 0, 'H dan W harus kelipatan 64'
    recon, total, shapes, h = [], 0, [], hashlib.sha256()
    for i in range(0, T, batch):
        x = to_tensor(frames[i:i + batch]).to(device)
        enc = codec.compress(x)
        total += compressed_bytes(enc['strings'])
        shapes.append([int(v) for v in enc['shape']])
        for part in enc['strings']:
            for s in part:
                h.update(s)
        recon.append(to_uint8(codec.decompress(enc['strings'], enc['shape'])['x_hat']))
    recon = np.concatenate(recon)
    assert recon.shape == frames.shape and recon.dtype == np.uint8
    meta = {'frames': int(T), 'height': int(H), 'width': int(W), 'batch': int(batch),
            'shape_per_batch': shapes, 'compressed_bytes': int(total),
            'bpp': total * 8 / (T * H * W), 'strings_sha256': h.hexdigest()}
    return recon, meta


def psnr(a, b):
    mse = np.mean((a.astype(np.float64) - b.astype(np.float64)) ** 2)
    return float(10 * np.log10(255.0 ** 2 / max(mse, 1e-10)))


def ssim(a, b, chunk=16):
    '''SSIM rata-rata (jendela Gaussian 11x11, sigma 1.5; Wang et al. 2004; tepi direfleksikan) antara dua array uint8 (T,H,W,3).
    Dihitung per kanal lalu dirata-rata; per potongan frame agar hemat memori.'''
    C1, C2 = (0.01 * 255.0) ** 2, (0.03 * 255.0) ** 2
    tot, n = 0.0, 0
    for i in range(0, len(a), chunk):
        x, y = a[i:i + chunk].astype(np.float64), b[i:i + chunk].astype(np.float64)
        g = lambda z: gaussian_filter(z, sigma=(0, 1.5, 1.5, 0), truncate=3.5, mode='reflect')
        mx, my = g(x), g(y)
        sxx, syy, sxy = g(x * x) - mx ** 2, g(y * y) - my ** 2, g(x * y) - mx * my
        s = ((2 * mx * my + C1) * (2 * sxy + C2)) / ((mx ** 2 + my ** 2 + C1) * (sxx + syy + C2))
        tot += float(s.sum()); n += s.size
    return tot / n


print(f'Neural Codec {CODEC_NAME}, quality={CODEC_QUALITY} siap (pretrained, frozen, eval).')

In [ ]:
def read_video(path, max_frames=MAX_FRAMES, size=FRAME_SIZE):
    '''ffprobe (fps) + ffmpeg (frame) -> (frame uint8 (T,H,W,3) RGB, fps).'''
    H, W = size
    probe = subprocess.run(['ffprobe', '-v', 'error', '-select_streams', 'v:0', '-show_entries',
                            'stream=r_frame_rate', '-of', 'csv=p=0', str(path)], capture_output=True, text=True).stdout.strip()
    try:
        a, b = probe.split('/'); fps = float(a) / float(b)
        fps = fps if 1 <= fps <= 240 else 25.0
    except Exception:
        fps = 25.0
    cmd = ['ffmpeg', '-v', 'error', '-i', str(path), '-vf', f'scale={W}:{H}:flags=area']
    cmd += ['-frames:v', str(max_frames)] if max_frames else []
    raw = np.frombuffer(subprocess.run(cmd + ['-f', 'rawvideo', '-pix_fmt', 'rgb24', '-'], capture_output=True).stdout, np.uint8)
    n = raw.size // (H * W * 3)
    assert n > 0, f'tidak ada frame terbaca dari {path}'
    frames = raw[:n * H * W * 3].reshape(n, H, W, 3).copy()
    assert frames.shape[1:] == (H, W, 3), f'ukuran frame tidak sesuai: {frames.shape}'
    return frames, fps


def save_lossless(frames, path, fps):
    '''Arsip lossless FFV1 (hanya penyimpanan; BUKAN bagian pengujian kompresi). Scratch di /content, lalu disalin atomic.'''
    T, H, W, _ = frames.shape
    with tempfile.TemporaryDirectory() as scratch:
        tmp = Path(scratch) / 'archive.mkv'
        cmd = ['ffmpeg', '-y', '-v', 'error', '-f', 'rawvideo', '-pix_fmt', 'rgb24', '-s', f'{W}x{H}', '-r', str(fps),
               '-i', '-', '-c:v', 'ffv1', '-pix_fmt', 'gbrp', str(tmp)]
        res = subprocess.run(cmd, input=np.ascontiguousarray(frames).tobytes(), capture_output=True)
        assert res.returncode == 0, res.stderr.decode(errors='ignore')[-300:]
        atomic_copy(tmp, path)

## Ekstraksi frame

Video dipindai **deterministik** (sorted) dari `ROOT/test`, frame disimpan persistent di `ROOT/frames_tmp/<run_id>/`.
Scratch yang benar-benar disposable memakai `tempfile.TemporaryDirectory()` di `/content` dan tidak dicampur dengan folder di atas.

In [ ]:
def scan_dataset():
    exts = ('.mp4', '.avi', '.mov', '.mkv')
    vids = sorted(p for p in DATASET_DIR.iterdir() if p.is_file() and p.suffix.lower() in exts and '_wm' not in p.name.lower())
    assert vids, f'tidak ada video di {DATASET_DIR}'
    assert len({p.stem for p in vids}) == len(vids), 'nama video (tanpa ekstensi) harus unik'
    return vids[:MAX_VIDEOS]


video_files = scan_dataset()
VIDEO_NAMES = [p.name for p in video_files]
ckpt_frames = Checkpoint('frames').load()
if ckpt_frames.state.get('videos') not in (None, VIDEO_NAMES):
    raise CheckpointError('daftar video di dataset berbeda dari yang tercatat di checkpoint frames')
ckpt_frames.state['videos'] = VIDEO_NAMES
ckpt_run.commit(stage='DATASET_READY')
print(len(VIDEO_NAMES), 'video akan diuji')

fps_of = {}
for idx, p in enumerate(tqdm(video_files, desc='Ekstraksi frame')):
    done = ckpt_frames.state['done'].get(p.name)
    if done:
        verify_entry(done)                                   # resume: artifact harus ada & valid, kalau tidak -> FAIL
    else:
        frames, fps = read_video(p)
        art = FRAMES_DIR / f'{p.stem}.npy'
        atomic_save_npy(art, frames)                         # 1) tulis artifact
        done = {'artifacts': {'frames': make_artifact(art, frames.shape)}, 'fps': fps, 'frame_count': int(len(frames))}
        verify_entry(done)                                   # 2) verifikasi
        ckpt_frames.commit(stage='FRAMES_EXTRACTED', status='IN_PROGRESS', video=p.name, entry=done,
                           video_index=idx, frame_index=len(frames) - 1)   # 3) baru update checkpoint
    fps_of[p.name] = done['fps']
ckpt_frames.commit(stage='FRAMES_EXTRACTED', status='DONE')
ckpt_run.commit(stage='FRAMES_EXTRACTED')
print('Frame siap di', FRAMES_DIR)

## Payload `"sabila"` -> 48 bit

Tanpa ECC (tidak ada repetisi 3x, tidak ada payload 144 bit).

In [ ]:
def text_to_bits(text):
    return np.array([int(b) for ch in text.encode() for b in f'{ch:08b}'], np.int64)


def bits_to_text(bits):
    s = ''.join(str(int(b)) for b in bits)
    return bytes(int(s[i:i + 8], 2) for i in range(0, len(s), 8)).decode('utf-8', errors='replace')


def bits_str(bits):
    s = ''.join(str(int(b)) for b in bits)
    return ' '.join(s[i:i + 8] for i in range(0, len(s), 8))


PAYLOAD = text_to_bits(PAYLOAD_TEXT)
N_BITS = len(PAYLOAD)
assert len(PAYLOAD) == 48
assert N_BITS == GRID[0] * GRID[1], 'payload harus 48 bit = jumlah blok grid'
assert ''.join(map(str, PAYLOAD)) == PAYLOAD_BITS_STR
assert bits_to_text(PAYLOAD) == PAYLOAD_TEXT
print(f'Watermark asli : "{PAYLOAD_TEXT}"')
print(f'Payload ({N_BITS} bit): {bits_str(PAYLOAD)}')

## Pola deterministik dan penyisipan pixel

**Layout (eksplisit):** frame `192 x 256` dibagi grid `6 x 8` = 48 blok **menutupi seluruh frame**; tiap blok `32 x 32` pixel
(= 4x4 sel pola `8 x 8` pixel). Blok `b_i` memetakan bit ke-`i`, urutan row-major (kiri -> kanan, lalu turun baris). Kotak tiap blok: `block_boxes()` (Bagian Audit).

Pola ±1 berimbang, ditentukan **hanya oleh `(KEY, frame_index, bit_index)`** lewat `np.random.default_rng([KEY, frame, bit])`, jadi tidak bergantung pada RNG global.

`I'(x,y) = clip(I(x,y) + STRENGTH * (2b - 1) * P(x,y))`, lalu `uint8`.

In [ ]:
def make_patterns(n_frames, H, W, grid=GRID, cell=CELL, key=KEY, frame_offset=0):
    '''Pola ±1 berimbang per (frame, bit) -> (T, 48, bh, bw). Deterministik dari (key, frame_index, bit_index).'''
    R, C = grid; bh, bw = H // R, W // C
    assert H % R == 0 and W % C == 0 and bh % cell == 0 and bw % cell == 0
    ch, cw = bh // cell, bw // cell
    assert (ch * cw) % 2 == 0, 'jumlah sel per blok harus genap agar pola berimbang'
    base = np.array([1.0] * (ch * cw // 2) + [-1.0] * (ch * cw // 2), np.float32)
    cells = np.empty((n_frames, R * C, ch, cw), np.float32)
    for t in range(n_frames):
        for i in range(R * C):
            cells[t, i] = np.random.default_rng([key, frame_offset + t, i]).permutation(base).reshape(ch, cw)
    return np.kron(cells, np.ones((1, 1, cell, cell), np.float32))


_PAT_CACHE = {}
def get_patterns(T, H, W, frame_offset=0):
    '''Pola untuk frame frame_offset .. frame_offset+T-1. Cache aman: pola frame t tidak bergantung pada jumlah frame.'''
    need = frame_offset + T
    if (H, W) not in _PAT_CACHE or len(_PAT_CACHE[(H, W)]) < need:
        _PAT_CACHE[(H, W)] = make_patterns(max(need, MAX_FRAMES), H, W)
    return _PAT_CACHE[(H, W)][frame_offset:need]


def embed_pixel_watermark(frames, bits=PAYLOAD, strength=STRENGTH):
    assert frames.dtype == np.uint8 and frames.ndim == 4
    T, H, W, _ = frames.shape; R, C = GRID; bh, bw = H // R, W // C
    pats = get_patterns(T, H, W)
    sgn = (np.asarray(bits) * 2 - 1).astype(np.float32)
    delta = (sgn[None, :, None, None] * pats).reshape(T, R, C, bh, bw).transpose(0, 1, 3, 2, 4).reshape(T, H, W)
    return np.clip(np.rint(frames.astype(np.float32) + strength * delta[..., None]), 0, 255).astype(np.uint8)

In [ ]:
ckpt_embed = Checkpoint('embed').load()
wm_names = {}
for idx, name in enumerate(tqdm(VIDEO_NAMES, desc='Sisip watermark')):
    fe = ckpt_frames.state['done'].get(name)
    if fe is None:
        raise CheckpointError(f'frame {name} belum diekstrak')
    stem = Path(name).stem
    done = ckpt_embed.state['done'].get(name)
    if done:
        verify_entry(done)
    else:
        frames = load_npy(verify_entry(fe), 'frames')
        wm = embed_pixel_watermark(frames)
        art_npy, art_mkv = DIRS['watermarked'] / f'{stem}_wm.npy', DIRS['watermarked'] / f'{stem}_wm.mkv'
        atomic_save_npy(art_npy, wm)
        save_lossless(wm, art_mkv, fe['fps'])               # arsip FFV1 (lossless)
        done = {'artifacts': {'watermarked': make_artifact(art_npy, wm.shape), 'archive': make_artifact(art_mkv)},
                'frame_count': int(len(wm))}
        verify_entry(done)
        ckpt_embed.commit(stage='WATERMARK_EMBEDDED', status='IN_PROGRESS', video=name, entry=done,
                          video_index=idx, frame_index=len(wm) - 1)
ckpt_embed.commit(stage='WATERMARK_EMBEDDED', status='DONE')
ckpt_run.commit(stage='WATERMARK_EMBEDDED')
print(len(VIDEO_NAMES), 'video ber-watermark tersedia di', DIRS['watermarked'])

## Neural Codec: compress() -> decompress()

Yang dipersist: frame hasil rekonstruksi (`reconstructed/`) dan metadata compressed representation (`compressed/`: ukuran byte, shape, BPP, SHA-256 `strings`).
Compress + decompress ditulis sebagai satu unit; checkpoint baru naik ke `CODEC_RECONSTRUCTED` setelah kedua artifact tertulis dan terverifikasi.

In [ ]:
ckpt_codec = Checkpoint('codec').load()
codec_meta = {}
for idx, name in enumerate(tqdm(VIDEO_NAMES, desc='Neural Codec (ber-watermark)')):
    fe = ckpt_embed.state['done'].get(name)
    if fe is None:
        raise CheckpointError(f'watermark {name} belum disisipkan')
    stem = Path(name).stem
    done = ckpt_codec.state['done'].get(name)
    if done:
        verify_entry(done)                                   # checkpoint bilang selesai -> artifact WAJIB ada & valid
    else:
        wm = load_npy(verify_entry(fe), 'watermarked')
        recon, meta = codec_roundtrip(wm)
        art_rec, art_cmp = DIRS['reconstructed'] / f'{stem}_rec.npy', DIRS['compressed'] / f'{stem}_compressed.json'
        atomic_save_npy(art_rec, recon)
        atomic_write_json(art_cmp, meta)
        done = {'artifacts': {'reconstructed': make_artifact(art_rec, recon.shape), 'compressed': make_artifact(art_cmp)},
                'bpp': meta['bpp'], 'compressed_bytes': meta['compressed_bytes'], 'frame_count': int(len(recon))}
        verify_entry(done)
        ckpt_codec.commit(stage='CODEC_RECONSTRUCTED', status='IN_PROGRESS', video=name, entry=done,
                          video_index=idx, frame_index=len(recon) - 1)
    codec_meta[name] = done
ckpt_codec.commit(stage='CODEC_RECONSTRUCTED', status='DONE')
ckpt_run.commit(stage='CODEC_RECONSTRUCTED')
print(f'bpp rata-rata: {np.mean([m["bpp"] for m in codec_meta.values()]):.3f}')

## Ekstraksi watermark dari pixel (pada frame hasil rekonstruksi)

Untuk tiap blok: korelasi nilai pixel (rata-rata kanal) dengan pola ±1 bit tersebut, dirata-ratakan antar frame. `score > 0 -> 1`, selain itu `0`.

In [ ]:
def extract_pixel_watermark(frames, return_score=False, frame_offset=0):
    T, H, W, _ = frames.shape; R, C = GRID; bh, bw = H // R, W // C
    pats = get_patterns(T, H, W, frame_offset)   # frame_offset = indeks frame pertama pada video asal
    g = frames.astype(np.float32).mean(-1)
    blocks = g.reshape(T, R, bh, C, bw).transpose(0, 1, 3, 2, 4).reshape(T, R * C, bh, bw)
    score = (blocks * pats).mean((2, 3)).mean(0)             # korelasi per bit, dirata-rata antar frame
    bits = (score > 0).astype(np.int64)                      # score > 0 -> 1 ; score <= 0 -> 0
    return (bits, score) if return_score else bits


read_bits, read_scores = {}, {}
for name in tqdm(VIDEO_NAMES, desc='Ekstraksi pixel'):
    recon = load_npy(verify_entry(ckpt_codec.state['done'][name]), 'reconstructed')    # BUKAN frame asli/ber-watermark
    read_bits[name], read_scores[name] = extract_pixel_watermark(recon, return_score=True)
ckpt_run.commit(stage='WATERMARK_EXTRACTED')

## Evaluasi: BER, Bit Accuracy, Deteksi, PSNR, BPP

- `BER = bit_salah / 48`, `Bit Accuracy = bit_benar / 48`.
- **H0:** peluang sebuah bit cocok hanya karena kebetulan = 0.5 (binomial, n = 48). `p = binom.sf(k - 1, 48, 0.5)`, `k` = jumlah bit benar.
- `p <= 1e-4` -> **DETECTED**, selain itu **NOT DETECTED**.
- **PSNR watermark** dan **SSIM watermark** = asli vs ber-watermark (distorsi embedding = ukuran imperceptibility). **PSNR codec** = ber-watermark vs rekonstruksi (distorsi Neural Codec).

In [ ]:
MIN_MATCH = next(k for k in range(N_BITS + 1) if binom.sf(k - 1, N_BITS, 0.5) <= P_FALSE_ALARM)
print(f'DETECTED jika bit cocok >= {MIN_MATCH}/{N_BITS} (BER <= {1 - MIN_MATCH / N_BITS:.3f}); H0: P(bit cocok) = 0.5, p <= {P_FALSE_ALARM}')


def evaluate(bits_read):
    k = int((np.asarray(bits_read) == PAYLOAD).sum())
    p = float(binom.sf(k - 1, N_BITS, 0.5))
    return {'ber': 1 - k / N_BITS, 'bit_accuracy': k / N_BITS, 'bits_correct': k,
            'detection_p_value': p, 'detection_status': 'DETECTED' if p <= P_FALSE_ALARM else 'NOT DETECTED',
            'text_read': bits_to_text(bits_read)}


rows = []
for name in tqdm(VIDEO_NAMES, desc='Evaluasi'):
    orig = load_npy(verify_entry(ckpt_frames.state['done'][name]), 'frames')
    wm = load_npy(verify_entry(ckpt_embed.state['done'][name]), 'watermarked')
    rec = load_npy(verify_entry(ckpt_codec.state['done'][name]), 'reconstructed')
    rows.append({'run_id': RUN_ID, 'video': name, 'frame_count': int(len(orig)), 'payload': PAYLOAD_TEXT,
                 'codec': CODEC_NAME, 'codec_quality': CODEC_QUALITY, 'strength': STRENGTH, 'key': KEY,
                 **evaluate(read_bits[name]),
                 'psnr_watermark_db': psnr(orig, wm),        # asli vs ber-watermark
                 'ssim_watermark': ssim(orig, wm),           # asli vs ber-watermark
                 'psnr_codec_db': psnr(wm, rec),             # ber-watermark vs hasil Neural Codec
                 'bpp': codec_meta[name]['bpp']})

## False Positive Control

Frame **asli** (tanpa watermark) -> **jalur codec yang sama** (`codec_roundtrip`) -> ekstraksi pixel -> deteksi.
Mengukur apakah extractor "mendeteksi" `sabila` padahal tidak pernah ditanam.

In [ ]:
ckpt_fp = Checkpoint('false_positive').load()
rows_fp = []
for idx, name in enumerate(tqdm(VIDEO_NAMES, desc='False positive')):
    done = ckpt_fp.state['done'].get(name)
    if done:
        verify_entry(done)
        row = json.loads(Path(done['artifacts']['result']['path']).read_text(encoding='utf-8'))
    else:
        orig = load_npy(verify_entry(ckpt_frames.state['done'][name]), 'frames')
        rec_fp, meta_fp = codec_roundtrip(orig)               # tanpa embedding, codec path sama dengan test utama
        row = {'run_id': RUN_ID, 'video': name, 'frame_count': int(len(orig)), 'payload': PAYLOAD_TEXT,
               **evaluate(extract_pixel_watermark(rec_fp)), 'bpp': meta_fp['bpp']}
        art = DIRS['reports'] / f'fp_{Path(name).stem}.json'
        atomic_write_json(art, row)
        done = {'artifacts': {'result': make_artifact(art)}}
        verify_entry(done)
        ckpt_fp.commit(stage='WATERMARK_EXTRACTED', status='IN_PROGRESS', video=name, entry=done, video_index=idx)
    rows_fp.append(row)
ckpt_fp.commit(stage='WATERMARK_EXTRACTED', status='DONE')
n_fp = sum(r['detection_status'] == 'DETECTED' for r in rows_fp)
print(f'False positive rate: {n_fp / len(rows_fp):.2%} ({n_fp}/{len(rows_fp)} video asli salah terdeteksi "{PAYLOAD_TEXT}")')

## Metrics + manifest

`output/metrics_<run_id>.csv`, `output/metrics_fp_<run_id>.csv`, `output/manifest_<run_id>.json`.

In [ ]:
df = pd.DataFrame(rows)
df_fp = pd.DataFrame(rows_fp)
assert 'bpp' in df.columns

fmt = {'ber': '{:.4f}', 'bit_accuracy': '{:.4f}', 'detection_p_value': '{:.2e}', 'psnr_watermark_db': '{:.2f}', 'ssim_watermark': '{:.4f}',
       'psnr_codec_db': '{:.2f}', 'bpp': '{:.3f}'}
cols = ['video', 'frame_count', 'ber', 'bit_accuracy', 'detection_status', 'detection_p_value',
        'psnr_watermark_db', 'ssim_watermark', 'psnr_codec_db', 'bpp', 'text_read']
print('=== Ber-watermark -> Neural Codec -> ekstraksi pixel ===')
display(df[cols].style.format(fmt))
print(f"Rata-rata: BER={df.ber.mean():.4f} | Bit Accuracy={df.bit_accuracy.mean():.4f} | PSNR_WM={df.psnr_watermark_db.mean():.2f} dB | SSIM_WM={df.ssim_watermark.mean():.4f} | "
      f"PSNR_Codec={df.psnr_codec_db.mean():.2f} dB | bpp={df.bpp.mean():.3f} | DETECTED {(df.detection_status == 'DETECTED').sum()}/{len(df)}")
print('\n=== False positive (tanpa watermark) ===')
display(df_fp[['video', 'ber', 'bit_accuracy', 'detection_status', 'detection_p_value', 'text_read']].style.format(fmt))


def atomic_to_csv(frame, path):
    _atomic_write(path, lambda f: f.write(frame.to_csv(index=False).encode('utf-8')))


METRICS_CSV = OUT_ROOT / f'metrics_{RUN_ID}.csv'
METRICS_FP_CSV = OUT_ROOT / f'metrics_fp_{RUN_ID}.csv'
MANIFEST_JSON = OUT_ROOT / f'manifest_{RUN_ID}.json'
atomic_to_csv(df, METRICS_CSV)
atomic_to_csv(df_fp, METRICS_FP_CSV)

manifest = {
    'run_id': RUN_ID,
    'payload': PAYLOAD_TEXT,
    'payload_bits': PAYLOAD_BITS_STR,
    'dataset': {'dir': str(DATASET_DIR), 'videos': VIDEO_NAMES},
    'embedding_configuration': EMBED_CFG,
    'codec_configuration': CODEC_CFG,
    'frame_configuration': {'frame_size_hw': list(FRAME_SIZE), 'max_frames': MAX_FRAMES, 'grid': list(GRID), 'cell': CELL,
                            'block_pixels': [FRAME_SIZE[0] // GRID[0], FRAME_SIZE[1] // GRID[1]]},
    'detection': {'null_hypothesis': 'P(bit cocok) = 0.5', 'p_false_alarm': P_FALSE_ALARM, 'min_match': MIN_MATCH, 'n_bits': N_BITS},
    'summary': {'videos': len(df), 'mean_ber': float(df.ber.mean()), 'mean_bit_accuracy': float(df.bit_accuracy.mean()),
                'detected': int((df.detection_status == 'DETECTED').sum()), 'mean_bpp': float(df.bpp.mean()),
                'mean_psnr_watermark_db': float(df.psnr_watermark_db.mean()), 'mean_ssim_watermark': float(df.ssim_watermark.mean()), 'mean_psnr_codec_db': float(df.psnr_codec_db.mean()),
                'false_positive_rate': float((df_fp.detection_status == 'DETECTED').mean())},
    'output_directories': {k: str(v) for k, v in DIRS.items()} | {'frames_tmp': str(FRAMES_DIR)},
    'output_files': {'metrics_csv': str(METRICS_CSV), 'metrics_fp_csv': str(METRICS_FP_CSV)},
    'checkpoint_files': {n: str(CKPT_DIR / f'{n}.json') for n in ['run', 'frames', 'embed', 'codec', 'false_positive']},
    'timestamp': utc_now(),
}
atomic_write_json(MANIFEST_JSON, manifest)

# verifikasi hasil tulis sebelum checkpoint dinaikkan ke EVALUATED
assert len(pd.read_csv(METRICS_CSV)) == len(df) and len(pd.read_csv(METRICS_FP_CSV)) == len(df_fp)
assert json.loads(MANIFEST_JSON.read_text(encoding='utf-8'))['run_id'] == RUN_ID
ckpt_run.commit(stage='EVALUATED', status='EVALUATED')
print('Metrics :', METRICS_CSV)
print('Manifest:', MANIFEST_JSON)

## Visual / Audit

Bagian ini menjawab: apakah Neural Codec benar-benar mengompresi, di blok pixel mana tiap bit payload berada, bagaimana nilai pixel berubah,
apakah watermark terbaca kembali dari frame **hasil rekonstruksi**, dan apakah watermark cukup imperceptible. Semua nilai berasal dari artifact run ini
(tanpa data dummy). Gambar *selisih* hanya dipakai sebagai **audit/debug**; penilaian imperceptibility memakai frame asli apa adanya + PSNR/SSIM + strength sweep.

In [ ]:
def block_boxes(H, W, grid=GRID):
    """Kotak (y0, y1, x0, x1) tiap blok, row-major b0..b47. Rumus sama dengan embed_pixel_watermark: bh = H//R, bw = W//C."""
    R, C = grid; bh, bw = H // R, W // C
    return [(r * bh, (r + 1) * bh, c * bw, (c + 1) * bw) for r in range(R) for c in range(C)]


def load_triplet(name):
    """(asli, ber-watermark, hasil rekonstruksi codec) uint8 (T,H,W,3); semua dari artifact yang sudah diverifikasi."""
    orig = load_npy(verify_entry(ckpt_frames.state['done'][name]), 'frames')
    wm = load_npy(verify_entry(ckpt_embed.state['done'][name]), 'watermarked')
    rec = load_npy(verify_entry(ckpt_codec.state['done'][name]), 'reconstructed')
    return orig, wm, rec


def _expected_wm(orig, bits=PAYLOAD, strength=STRENGTH):
    """Rekonstruksi independen: tempel blok b_i satu per satu di kotaknya (tanpa trik reshape)."""
    T, H, W, _ = orig.shape
    pats = get_patterns(T, H, W)
    delta = np.zeros((T, H, W), np.float32)
    for i, (y0, y1, x0, x1) in enumerate(block_boxes(H, W)):
        delta[:, y0:y1, x0:x1] = strength * (2 * int(bits[i]) - 1) * pats[:, i]
    return np.clip(np.rint(orig.astype(np.float32) + delta[..., None]), 0, 255).astype(np.uint8), pats


def _draw_grid(ax, boxes, labels, ec='yellow', fs=7):
    for i, (y0, y1, x0, x1) in enumerate(boxes):
        ax.add_patch(plt.Rectangle((x0 - .5, y0 - .5), x1 - x0, y1 - y0, fill=False, ec=ec, lw=1))
        ax.text((x0 + x1) / 2 - .5, (y0 + y1) / 2 - .5, labels[i], ha='center', va='center', fontsize=fs,
                bbox=dict(fc='white', alpha=.75, pad=1, ec='none'))


def visualize_watermark_location(orig, wm, frame_idx=0, bits=PAYLOAD, name='', save_path=None):
    """AUDIT/DEBUG: lokasi & urutan bit pada satu frame. Panel selisih diberi skala warna (diperbesar) dan BUKAN tampilan sebenarnya."""
    T, H, W, _ = orig.shape; R, C = GRID
    boxes = block_boxes(H, W)
    o, w = orig[frame_idx], wm[frame_idx]
    diff = w.astype(np.float32).mean(-1) - o.astype(np.float32).mean(-1)
    m = max(float(np.abs(diff).max()), 1.0)
    lab_bit = [f'{i}:{int(bits[i])}' for i in range(len(bits))]
    lab_ord = [f'b{i}\n={int(bits[i])}' for i in range(len(bits))]
    fig, ax = plt.subplots(2, 2, figsize=(15, 11))
    ax[0, 0].imshow(o, interpolation='nearest'); ax[0, 0].set_title(f'Frame Asli (frame {frame_idx})')
    ax[0, 1].imshow(w, interpolation='nearest'); _draw_grid(ax[0, 1], boxes, lab_bit)
    ax[0, 1].set_title(f'Frame Setelah Watermark (frame {frame_idx}) | label = "nomor blok:bit"')
    im = ax[1, 0].imshow(diff, cmap='seismic', vmin=-m, vmax=m); _draw_grid(ax[1, 0], boxes, lab_bit, ec='black')
    ax[1, 0].set_title(f'[AUDIT/DEBUG - bukan tampilan sebenarnya] selisih pixel, skala ±{m:.0f} level')
    plt.colorbar(im, ax=ax[1, 0], fraction=0.046, pad=0.02)
    ax[1, 1].imshow(np.asarray(bits).reshape(R, C), cmap='Greens', vmin=0, vmax=1.6)
    for i in range(len(bits)):
        ax[1, 1].text(i % C, i // C, lab_ord[i], ha='center', va='center', fontsize=9)
    ax[1, 1].set_xticks(range(C)); ax[1, 1].set_yticks(range(R))
    ax[1, 1].set_title('Urutan blok row-major: b0 -> b47 (hijau tua = bit 1)')
    for a in ax.ravel()[:3]:
        a.axis('off')
    fig.suptitle(f'{name} | payload "{PAYLOAD_TEXT}" | grid {R} x {C}', fontsize=13)
    plt.tight_layout(rect=(0, 0, 1, 0.97))
    if save_path:
        fig.savefig(save_path, dpi=110)
    plt.show()


def check_watermark(orig, wm, bits=PAYLOAD, strength=STRENGTH, show_frames=5, verbose=True):
    """Validasi otomatis: lokasi, urutan, payload, dan keberadaan watermark di SETIAP frame."""
    T, H, W, _ = orig.shape; R, C = GRID; n = len(bits)
    exp, pats = _expected_wm(orig, bits, strength)
    same = (exp == wm).reshape(T, -1).all(1)
    dg = wm.astype(np.float32).mean(-1) - orig.astype(np.float32).mean(-1)
    boxes = block_boxes(H, W)
    corr = np.stack([(dg[:, y0:y1, x0:x1] * pats[:, i]).mean((1, 2)) for i, (y0, y1, x0, x1) in enumerate(boxes)], 1)
    energy = np.stack([np.abs(dg[:, y0:y1, x0:x1]).mean((1, 2)) for (y0, y1, x0, x1) in boxes], 1)
    planted = (corr > 0).astype(int)
    ok_bits = (planted == np.asarray(bits)[None]).all(1)
    ok_block = (energy > 0).all(1)
    checks = {
        'Setiap frame punya 48 blok': len(boxes) == R * C == n,
        'Posisi blok sama di tiap frame': bool(same.all()),
        'Urutan bit b0 -> b47 (row-major)': bool(ok_bits.all()),
        'Payload tiap frame = payload asli': bool(ok_bits.all()),
        'Tidak ada frame terlewat': bool(ok_block.all()) and len(wm) == len(orig),
        'Total bit ditanam = frame x 48': int((planted == np.asarray(bits)[None]).sum()) == T * n,
    }
    if verbose:
        print('[CHECK WATERMARK]')
        print(f'Jumlah frame: {T} | Payload: {PAYLOAD_TEXT} ({n} bit) | Grid: {R} x {C}')
        print(f'Total bit ditanam: {int((planted == np.asarray(bits)[None]).sum())} (harapan {T * n})')
        for t in range(min(show_frames, T)):
            print(f'Frame {t} -> {int((planted[t] == np.asarray(bits)).sum())}/{n} bit sesuai' + ('' if ok_bits[t] and same[t] else '  <-- TIDAK SESUAI'))
        for k, ok in checks.items():
            print(f'  [{"OK" if ok else "GAGAL"}] {k}')
    return checks, same, ok_bits


def audit_report(name, frame_idx=0):
    """Laporan audit satu video: lokasi blok, pixel sebelum/sesudah/selisih (pixel pertama + rata-rata RGB per blok),
    ukuran compressed representation, korelasi extractor, dan bit hasil decoding. Disimpan ke reports/audit_<video>_f<frame>.json"""
    stem = Path(name).stem
    orig, wm, rec = load_triplet(name)
    assert 0 <= frame_idx < len(orig), f'AUDIT_FRAME_IDX={frame_idx} di luar jumlah frame video ({len(orig)})'
    cmp_meta = json.loads(Path(ckpt_codec.state['done'][name]['artifacts']['compressed']['path']).read_text(encoding='utf-8'))
    bits_dec, score = extract_pixel_watermark(rec, return_score=True)
    T, H, W, _ = orig.shape
    blocks = []
    for i, (y0, y1, x0, x1) in enumerate(block_boxes(H, W)):
        ob, wb = orig[frame_idx, y0:y1, x0:x1].astype(int), wm[frame_idx, y0:y1, x0:x1].astype(int)
        d = wb - ob
        blocks.append({'block': i, 'box_y0y1x0x1': [y0, y1, x0, x1], 'bit': int(PAYLOAD[i]),
                       'pixel_before_rgb': ob[0, 0].tolist(), 'pixel_after_rgb': wb[0, 0].tolist(), 'pixel_diff_rgb': d[0, 0].tolist(),
                       'mean_rgb_before': ob.mean((0, 1)).tolist(), 'mean_rgb_after': wb.mean((0, 1)).tolist(),
                       'mean_abs_diff': float(np.abs(d).mean()),
                       'correlation_score': float(score[i]), 'decoded_bit': int(bits_dec[i])})
    report = {'run_id': RUN_ID, 'video': name, 'frame_idx': frame_idx, 'payload': PAYLOAD_TEXT, 'payload_bits': bits_str(PAYLOAD),
              'decoded_bits': bits_str(bits_dec), 'decoded_text': bits_to_text(bits_dec),
              'compressed_representation': {'bytes': cmp_meta['compressed_bytes'], 'bpp': cmp_meta['bpp'],
                                            'shape_per_batch': cmp_meta['shape_per_batch'], 'strings_sha256': cmp_meta['strings_sha256']},
              'reconstructed_frame_shape': list(rec[frame_idx].shape), 'blocks': blocks}
    atomic_write_json(DIRS['reports'] / f'audit_{stem}_f{frame_idx}.json', report)
    return report, orig, wm, rec


assert 0 <= AUDIT_VIDEO_INDEX < len(VIDEO_NAMES), 'AUDIT_VIDEO_INDEX di luar daftar video'
audit_name = VIDEO_NAMES[AUDIT_VIDEO_INDEX]
rep, orig_v, wm_v, rec_v = audit_report(audit_name, AUDIT_FRAME_IDX)
print(f'Video diaudit  : {audit_name} | frame {AUDIT_FRAME_IDX} | {len(orig_v)} frame')
print(f'Payload asli   : {rep["payload_bits"]}  ("{rep["payload"]}")')
print(f'Payload terbaca: {rep["decoded_bits"]}  ("{rep["decoded_text"]}")')
print(f'Compressed repr: {rep["compressed_representation"]["bytes"]} byte ({rep["compressed_representation"]["bpp"]:.3f} bpp)')

checks, same, ok_bits = check_watermark(orig_v, wm_v)
assert all(checks.values()), 'Ada ketidaksesuaian lokasi/urutan/payload watermark!'
rows_chk = []
for name in VIDEO_NAMES:
    o = load_npy(ckpt_frames.state['done'][name], 'frames'); w = load_npy(ckpt_embed.state['done'][name], 'watermarked')
    c, s, ob = check_watermark(o, w, verbose=False)
    rows_chk.append({'video': name, 'frames': len(o), 'bits_planted': len(o) * N_BITS, 'frames_ok': int((s & ob).sum()),
                     'all_checks': 'OK' if all(c.values()) else 'FAIL'})
df_chk = pd.DataFrame(rows_chk)
display(df_chk)
assert (df_chk.all_checks == 'OK').all(), 'audit watermark gagal pada sebagian video'

# tampilan lokasi (AUDIT/DEBUG: panel selisih diperbesar)
for t in sorted({0, len(orig_v) // 2, len(orig_v) - 1}):
    visualize_watermark_location(orig_v, wm_v, frame_idx=t, name=audit_name,
                                 save_path=DIRS['reports'] / f'location_{Path(audit_name).stem}_f{t}.png')

## Bukti Neural Codec: compress() -> strings -> decompress()

Cell di bawah memanggil `codec.compress()` dan `codec.decompress()` **langsung** pada satu frame ber-watermark dan menampilkan isi nyata compressed representation.
Bukti yang dihasilkan:

1. ukuran compressed = jumlah **byte seluruh `strings`** (semua bagian, bertingkat), bukan `len(strings)`;
2. **cross-decode**: `decompress()` pada bitstream frame A menghasilkan gambar A, bukan frame B. Jadi rekonstruksi memang ditentukan oleh `strings`;
3. kecocokan dengan artifact `reconstructed/` yang dipakai pipeline (satu batch dikompresi ulang lalu dibandingkan).

In [ ]:
@torch.no_grad()
def _encode_decode(frame_u8):
    """frame (1,H,W,3) uint8 -> compress() -> (strings, shape) -> decompress() -> rekonstruksi uint8."""
    enc = codec.compress(to_tensor(frame_u8).to(device))
    rec = to_uint8(codec.decompress(enc['strings'], enc['shape'])['x_hat'])
    return enc['strings'], enc['shape'], rec


def codec_proof(wm_video, rec_saved, frame_idx, other_frame=None):
    f = wm_video[frame_idx:frame_idx + 1]
    H, W = f.shape[1:3]
    strings, shape, rec = _encode_decode(f)
    parts = [{'part': k, 'items': len(p), 'bytes_each': [len(s) for s in p], 'bytes_total': int(sum(len(s) for s in p))}
             for k, p in enumerate(strings)]
    total = compressed_bytes(strings)                       # jumlah byte SEMUA string (nested), bukan len(strings)
    raw = H * W * 3
    out = {'model': CODEC_NAME, 'quality': CODEC_QUALITY, 'pretrained': CODEC_CFG['pretrained'],
           'frozen': bool(all(not p.requires_grad for p in codec.parameters()) and not codec.training),
           'n_parts': len(strings), 'n_strings': int(sum(len(p) for p in strings)), 'parts': parts,
           'compressed_bytes': int(total), 'raw_bytes': int(raw), 'compression_ratio': raw / total, 'bpp': total * 8 / (H * W),
           'latent_shape': [int(v) for v in shape], 'original_shape': list(f.shape[1:]), 'reconstructed_shape': list(rec.shape[1:]),
           'psnr_codec_db': psnr(f, rec), 'pixels_changed_by_codec': float((rec != f).any(-1).mean())}
    assert rec.shape == f.shape and out['compressed_bytes'] > 0
    # (2) cross-decode: bitstream frame A -> gambar A; bitstream frame B -> gambar B
    if other_frame is not None:
        s_b, sh_b, rec_b = _encode_decode(other_frame)
        rec_a_on_b = to_uint8(codec.decompress(strings, shape)['x_hat'])      # decode bitstream A, dibandingkan dengan frame B
        out['cross_decode'] = {'psnr_B_vs_decode(strings_B)': psnr(other_frame, rec_b),
                               'psnr_B_vs_decode(strings_A)': psnr(other_frame, rec_a_on_b),
                               'decode(strings_A)_equals_rec_A': bool(np.array_equal(rec_a_on_b, rec))}
        assert out['cross_decode']['psnr_B_vs_decode(strings_B)'] > out['cross_decode']['psnr_B_vs_decode(strings_A)'], \
            'rekonstruksi tidak ditentukan oleh strings -> codec tidak benar-benar dipakai!'
    # (3) kecocokan dengan artifact pipeline (satu batch yang sama)
    nb_ = min(CODEC_BATCH, len(wm_video))
    rec_chk, meta_chk = codec_roundtrip(wm_video[:nb_])
    out['vs_saved_reconstructed'] = {'frames': int(nb_), 'identical_pixel_fraction': float((rec_chk == rec_saved[:nb_]).mean()),
                                     'max_abs_diff': int(np.abs(rec_chk.astype(int) - rec_saved[:nb_].astype(int)).max()),
                                     'psnr_between_db': psnr(rec_chk, rec_saved[:nb_])}
    return out, f, rec, strings


other_f = None
if len(VIDEO_NAMES) > 1:                                    # frame dari VIDEO LAIN (isi gambar berbeda jauh)
    other_name = next(n for n in VIDEO_NAMES if n != audit_name)
    other_f = load_npy(verify_entry(ckpt_embed.state['done'][other_name]), 'watermarked')[0:1]
    if other_f.shape[1:] != wm_v.shape[1:]:
        other_f = None
proof, f_in, f_out, strings_v = codec_proof(wm_v, rec_v, AUDIT_FRAME_IDX, other_f)

print('Neural Codec')
print('-' * 30)
print(f"Model       : {proof['model']}")
print(f"Quality     : {proof['quality']}")
print(f"Pretrained  : {proof['pretrained']}  (dimuat dengan pretrained=True)")
print(f"Frozen      : {proof['frozen']}  (requires_grad=False untuk semua parameter, mode eval)")
print()
print(f'Compressed representation (frame {AUDIT_FRAME_IDX} dari {audit_name}):')
print(f"Number of strings : {proof['n_strings']} string byte dalam {proof['n_parts']} bagian (bagian ke-k = bitstream y / z)")
for p in proof['parts']:
    print(f"  bagian {p['part']}: {p['items']} string, ukuran per string = {p['bytes_each']} byte, subtotal {p['bytes_total']} byte")
print(f"Compressed bytes  : {proof['compressed_bytes']}  (jumlah byte seluruh strings)")
print(f"Raw bytes         : {proof['raw_bytes']}  -> rasio kompresi {proof['compression_ratio']:.1f}x")
print(f"BPP               : {proof['bpp']:.4f}")
print(f"Latent shape      : {proof['latent_shape']}")
print()
print(f"Original shape    : {proof['original_shape']}")
print(f"Reconstructed     : {proof['reconstructed_shape']}")
print(f"PSNR codec        : {proof['psnr_codec_db']:.2f} dB (watermarked vs hasil decompress)")
print(f"Pixel yang berubah oleh codec: {proof['pixels_changed_by_codec']:.1%}")
if 'cross_decode' in proof:
    cd = proof['cross_decode']
    print('\nCross-decode (bukti rekonstruksi berasal dari strings):')
    print(f"  PSNR frame B vs decode(strings_B): {cd['psnr_B_vs_decode(strings_B)']:.2f} dB")
    print(f"  PSNR frame B vs decode(strings_A): {cd['psnr_B_vs_decode(strings_A)']:.2f} dB   <- harus jauh lebih rendah")
else:
    print('\nCross-decode dilewati: hanya ada satu video di dataset.')
v = proof['vs_saved_reconstructed']
print(f"\nKonsistensi dgn artifact pipeline ({v['frames']} frame): {v['identical_pixel_fraction']:.2%} pixel identik, "
      f"selisih maks {v['max_abs_diff']} level, PSNR antar-hasil {v['psnr_between_db']:.2f} dB")
print('(Selisih kecil bisa muncul dari nondeterminisme GPU; ini informasi, bukan syarat lulus.)')

CODEC_PROOF_JSON = DIRS['reports'] / f'codec_proof_{Path(audit_name).stem}_f{AUDIT_FRAME_IDX}.json'
atomic_write_json(CODEC_PROOF_JSON, proof)

fig, ax = plt.subplots(1, 3, figsize=(16, 4.6), gridspec_kw={'width_ratios': [1, 0.8, 1]})
ax[0].imshow(f_in[0], interpolation='nearest'); ax[0].set_title('Watermarked Frame (input codec)'); ax[0].axis('off')
ax[1].axis('off'); ax[1].set_xlim(0, 1); ax[1].set_ylim(0, 1)
ax[1].annotate('', xy=(0.97, 0.5), xytext=(0.03, 0.5), arrowprops=dict(arrowstyle='->', lw=2))
ax[1].text(0.5, 0.88, 'codec.compress()', ha='center', fontsize=12, weight='bold')
ax[1].text(0.5, 0.64, f"Compressed Representation\n{proof['n_strings']} string, {proof['compressed_bytes']} byte\n"
                      f"{proof['bpp']:.3f} bpp | rasio {proof['compression_ratio']:.1f}x", ha='center', va='center', fontsize=10,
           bbox=dict(boxstyle='round', fc='#eef', ec='navy'))
ax[1].text(0.5, 0.32, 'codec.decompress()', ha='center', fontsize=12, weight='bold')
ax[2].imshow(f_out[0], interpolation='nearest'); ax[2].set_title(f"Reconstructed Frame (output decompress)\nPSNR {proof['psnr_codec_db']:.2f} dB"); ax[2].axis('off')
plt.tight_layout()
fig.savefig(DIRS['reports'] / f'codec_flow_{Path(audit_name).stem}_f{AUDIT_FRAME_IDX}.png', dpi=110)
plt.show()

## Mapping payload `"sabila"` -> blok pixel

Satu bit = satu **blok** (`bh x bw` = 32 x 32 pixel pada frame 192 x 256). Di dalam blok, pola ±1 disusun dari **sel `CELL x CELL` = 8 x 8 pixel** (4 x 4 sel per blok).
Satu baris grid (8 blok) = satu byte = satu karakter. Koordinat diambil dari `block_boxes()`, fungsi yang sama yang dipakai `check_watermark()` untuk
membuktikan bahwa keluaran `embed_pixel_watermark()` identik dengan penempelan blok-blok ini.

In [ ]:
assert len(PAYLOAD_TEXT.encode()) == len(PAYLOAD_TEXT), 'pemetaan karakter -> 8 bit mengasumsikan teks ASCII'
H_v, W_v = orig_v.shape[1:3]
bits_dec_v, score_dec_v = extract_pixel_watermark(rec_v, return_score=True)


def payload_mapping_table(H, W, bits_read=None, scores=None):
    """Tabel payload -> blok pixel (koordinat dari block_boxes). Bila bits_read diberikan, tambahkan bit terbaca + skor korelasi."""
    rows_m = []
    for i, (y0, y1, x0, x1) in enumerate(block_boxes(H, W)):
        r_, c_ = divmod(i, GRID[1])
        row = {'bit': i, 'character': PAYLOAD_TEXT[i // 8], 'bit_in_char': i % 8, 'value': int(PAYLOAD[i]),
               'block (r,c)': f'({r_},{c_})', 'pixel_region': f'y {y0}:{y1}, x {x0}:{x1}', 'pixels': (y1 - y0) * (x1 - x0)}
        if bits_read is not None:
            row.update({'extracted': int(bits_read[i]), 'score': float(scores[i]), 'match': bool(bits_read[i] == PAYLOAD[i])})
        rows_m.append(row)
    return pd.DataFrame(rows_m)


chk_m, _, _ = check_watermark(orig_v, wm_v, verbose=False)
assert all(chk_m.values()), 'mapping tidak identik dengan hasil embedding'
df_map = payload_mapping_table(H_v, W_v, bits_dec_v, score_dec_v)
MAPPING_CSV = DIRS['reports'] / f'payload_mapping_{Path(audit_name).stem}.csv'
atomic_to_csv(df_map, MAPPING_CSV)
print(f'Payload "{PAYLOAD_TEXT}" = {len(PAYLOAD_TEXT.encode())} byte x 8 = {N_BITS} bit; grid {GRID[0]} x {GRID[1]} = {GRID[0] * GRID[1]} blok; '
      f'blok = {H_v // GRID[0]} x {W_v // GRID[1]} px (= {CELL} x {CELL} px per sel pola)')
with pd.option_context('display.max_rows', None):
    display(df_map)

boxes_m = block_boxes(H_v, W_v)
fig, ax = plt.subplots(figsize=(13, 10))
ax.imshow(wm_v[AUDIT_FRAME_IDX], interpolation='nearest')
_draw_grid(ax, boxes_m, [f'#{i}\n{int(PAYLOAD[i])}' for i in range(N_BITS)], ec='yellow', fs=9)
for r_ in range(GRID[0]):                                   # satu baris grid = satu karakter payload
    ya, yb = boxes_m[r_ * GRID[1]][0], boxes_m[r_ * GRID[1]][1]
    ax.text(-4, (ya + yb) / 2, f"'{PAYLOAD_TEXT[r_]}'", ha='right', va='center', fontsize=15, color='navy')
ax.set_xlim(-34, W_v - 0.5); ax.axis('off')
ax.set_title(f'Mapping payload "{PAYLOAD_TEXT}" pada frame ber-watermark (frame {AUDIT_FRAME_IDX}, {audit_name}) | label: #indeks bit / nilai bit')
plt.tight_layout()
fig.savefig(DIRS['reports'] / f'payload_mapping_{Path(audit_name).stem}_f{AUDIT_FRAME_IDX}.png', dpi=110)
plt.show()

## Audit pixel: sebelum / sesudah / delta

Tabel lengkap berisi **semua pixel semua blok** pada frame audit: `48 blok x (32 x 32) pixel x 3 kanal`. Satu blok = 1024 pixel = 3072 nilai kanal;
satu sel pola 8 x 8 = 64 pixel = 192 nilai kanal (kolom `cell_r`, `cell_c`). Nilai diambil dari frame asli dan frame ber-watermark yang diproses, lalu
`delta` aktual dibandingkan dengan rumus `STRENGTH * (2b - 1) * pola` (dengan memperhitungkan clipping 0..255). Seluruh tabel disimpan ke `reports/pixel_audit_*.csv`.

In [ ]:
def pixel_audit_frame(orig, wm, frame_idx, bits=PAYLOAD, strength=STRENGTH):
    """DataFrame semua pixel semua blok pada satu frame: koordinat, nilai RGB sebelum/sesudah, delta aktual, delta menurut rumus."""
    T, H, W, _ = orig.shape
    pats = get_patterns(1, H, W, frame_offset=frame_idx)[0]          # (48, bh, bw): pola frame ini, sama dengan embedding
    o, w = orig[frame_idx].astype(np.int16), wm[frame_idx].astype(np.int16)
    parts = []
    for i, (y0, y1, x0, x1) in enumerate(block_boxes(H, W)):
        yy, xx = np.mgrid[y0:y1, x0:x1]
        ob, wb, pb = o[y0:y1, x0:x1], w[y0:y1, x0:x1], pats[i]
        d = {'block': i, 'bit_index': i, 'expected_bit': int(bits[i]), 'y': yy.ravel(), 'x': xx.ravel(),
             'cell_r': ((yy - y0) // CELL).ravel(), 'cell_c': ((xx - x0) // CELL).ravel(), 'pattern': pb.ravel().astype(np.int8),
             'expected_delta': (strength * (2 * int(bits[i]) - 1) * pb).ravel().astype(np.int16)}
        for k, ch in enumerate('RGB'):
            d[f'{ch}_before'] = ob[..., k].ravel(); d[f'{ch}_after'] = wb[..., k].ravel(); d[f'd{ch}'] = (wb[..., k] - ob[..., k]).ravel()
        parts.append(pd.DataFrame(d))
    df_ = pd.concat(parts, ignore_index=True)
    ok, clipped = np.ones(len(df_), bool), np.zeros(len(df_), bool)
    for ch in 'RGB':                                                   # delta harapan setelah clip [0,255]
        b, e = df_[f'{ch}_before'].to_numpy().astype(int), df_['expected_delta'].to_numpy().astype(int)
        ok &= df_[f'd{ch}'].to_numpy() == (np.clip(b + e, 0, 255) - b)
        clipped |= (b + e < 0) | (b + e > 255)
    df_['clipped'], df_['delta_matches_formula'] = clipped, ok
    df_['d_mean'] = (df_.dR + df_.dG + df_.dB) / 3
    df_['abs_delta_max'] = np.maximum.reduce([df_.dR.abs(), df_.dG.abs(), df_.dB.abs()])
    return df_


def pixel_audit_summary(df_):
    df_ = df_.assign(d_x_pattern=df_.d_mean * df_.pattern)
    s = df_.groupby('block').agg(
        bit_index=('bit_index', 'first'), expected_bit=('expected_bit', 'first'), pixels=('y', 'size'),
        y0=('y', 'min'), y_max=('y', 'max'), x0=('x', 'min'), x_max=('x', 'max'),
        R_before=('R_before', 'mean'), G_before=('G_before', 'mean'), B_before=('B_before', 'mean'),
        R_after=('R_after', 'mean'), G_after=('G_after', 'mean'), B_after=('B_after', 'mean'),
        dR_mean=('dR', 'mean'), dG_mean=('dG', 'mean'), dB_mean=('dB', 'mean'),
        max_abs_delta=('abs_delta_max', 'max'), clipped_px=('clipped', 'sum'),
        formula_mismatch=('delta_matches_formula', lambda x: int((~x).sum())), corr_with_pattern=('d_x_pattern', 'mean'))
    s['planted_bit'] = (s.corr_with_pattern > 0).astype(int)
    return s.reset_index()


df_px = pixel_audit_frame(orig_v, wm_v, AUDIT_FRAME_IDX)
df_blk = pixel_audit_summary(df_px)
assert df_px['delta_matches_formula'].all(), 'delta aktual tidak sama dengan rumus embedding pada sebagian pixel'
assert (df_blk.planted_bit == df_blk.expected_bit).all(), 'bit yang tertanam pada pixel tidak sama dengan payload'
PIXEL_AUDIT_CSV = DIRS['reports'] / f'pixel_audit_{Path(audit_name).stem}_f{AUDIT_FRAME_IDX}.csv'
atomic_to_csv(df_px, PIXEL_AUDIT_CSV)
n_clip = int(df_px.clipped.sum())
print(f'{len(df_px)} pixel = {df_px.block.nunique()} blok x {len(df_px) // df_px.block.nunique()} pixel; {len(df_px) * 3} nilai kanal diaudit '
      f'(CSV lengkap: {PIXEL_AUDIT_CSV.name})')
print(f'Delta aktual == rumus STRENGTH*(2b-1)*pola pada semua pixel (clipping 0/255 diperhitungkan; {n_clip} pixel kena clipping).')
print('Bit yang tertanam (korelasi delta x pola > 0) sama dengan payload pada 48/48 blok.\n')
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(df_blk.style.format({c: '{:.2f}' for c in df_blk.columns if c.endswith(('_before', '_after', '_mean', 'pattern'))}))


def show_block(block=AUDIT_BLOCK):
    """Tabel lengkap nilai pixel satu blok (scroll). Semua 1024 pixel x RGB dapat diperiksa."""
    sub_ = df_px[df_px.block == block]
    ya, yb, xa, xb = block_boxes(*orig_v.shape[1:3])[block]
    print(f'Blok #{block} | bit={int(PAYLOAD[block])} (karakter "{PAYLOAD_TEXT[block // 8]}") | blok ({block // GRID[1]},{block % GRID[1]}) | '
          f'y {ya}:{yb}, x {xa}:{xb} | {len(sub_)} pixel = {len(sub_) * 3} nilai kanal')
    display(HTML('<div style="max-height:420px;overflow:auto;border:1px solid #ccc">' + sub_.drop(columns=['block']).to_html(index=False) + '</div>'))


show_block(AUDIT_BLOCK)
cell_sub = df_px[(df_px.block == AUDIT_BLOCK) & (df_px.cell_r == 0) & (df_px.cell_c == 0)]
assert len(cell_sub) == CELL * CELL
print(f'\nSel 8 x 8 pertama blok #{AUDIT_BLOCK}: {len(cell_sub)} pixel = {len(cell_sub) * 3} nilai kanal (pola = {int(cell_sub.pattern.iloc[0]):+d})')
display(HTML('<div style="max-height:300px;overflow:auto;border:1px solid #ccc">' + cell_sub.drop(columns=['block']).to_html(index=False) + '</div>'))
try:                                                          # opsional: pilih blok lain secara interaktif
    import ipywidgets as widgets
    widgets.interact(show_block, block=widgets.IntSlider(min=0, max=N_BITS - 1, step=1, value=AUDIT_BLOCK))
except Exception as e:
    print('Selector interaktif tidak tersedia:', type(e).__name__, '- ubah AUDIT_BLOCK lalu jalankan ulang cell ini.')

## Bukti watermark terbaca kembali (dari frame hasil rekonstruksi)

Hasil ekstraksi dihitung dari frame hasil `decompress()`, tanpa mengetahui payload. Payload dipakai hanya **sesudahnya** untuk menghitung BER.
Dua tingkat ditampilkan: **video-level** (skor korelasi dirata-rata seluruh frame, sama dengan metrik utama) dan **per-frame**. Kontrol negatif:
ekstraksi dari frame asli tanpa watermark.

In [ ]:
def show_result(title, res, bits_read):
    print(f'--- {title} ---')
    print(f'Expected payload : {PAYLOAD_TEXT}')
    print(f'Extracted payload: {res["text_read"]!r}')
    print(f'Expected bits    : {bits_str(PAYLOAD)}')
    print(f'Extracted bits   : {bits_str(bits_read)}')
    print(f'BER              : {res["ber"]:.4f}  ({N_BITS - res["bits_correct"]} bit salah dari {N_BITS})')
    print(f'Bit Accuracy     : {res["bit_accuracy"]:.4f}')
    print(f'Detection        : {"PASS" if res["detection_status"] == "DETECTED" else "FAIL"}  (p = {res["detection_p_value"]:.2e}, ambang {P_FALSE_ALARM})\n')


bits_v, score_v = extract_pixel_watermark(rec_v, return_score=True)         # frame HASIL REKONSTRUKSI, seluruh frame video
res_v = evaluate(bits_v)
per_frame = []
for t in range(len(rec_v)):
    b_t, s_t = extract_pixel_watermark(rec_v[t:t + 1], return_score=True, frame_offset=t)
    per_frame.append({'frame': t, 'score': s_t, 'bits': b_t, **evaluate(b_t)})
assert np.allclose(np.mean([p['score'] for p in per_frame], axis=0), score_v, atol=1e-4), 'skor video-level != rata-rata skor per-frame'
res_f = per_frame[AUDIT_FRAME_IDX]
bits_ctl = extract_pixel_watermark(orig_v)                                  # kontrol negatif: frame asli tanpa watermark
res_ctl = evaluate(bits_ctl)

show_result(f'VIDEO-LEVEL ({len(rec_v)} frame rata-rata) - {audit_name}', res_v, bits_v)
show_result(f'PER-FRAME (frame {AUDIT_FRAME_IDX} saja)', res_f, res_f['bits'])
show_result('KONTROL NEGATIF: frame asli (tanpa watermark, tanpa codec)', res_ctl, bits_ctl)

pf = pd.DataFrame([{k: p[k] for k in ('frame', 'ber', 'bit_accuracy', 'detection_status')} for p in per_frame])
print(f'Per-frame: BER rata-rata {pf.ber.mean():.4f} (min {pf.ber.min():.4f}, maks {pf.ber.max():.4f}); '
      f'{(pf.detection_status == "DETECTED").sum()}/{len(pf)} frame terdeteksi sendiri-sendiri; video-level BER {res_v["ber"]:.4f}.')
df_bits = pd.DataFrame({'bit': range(N_BITS), 'character': [PAYLOAD_TEXT[i // 8] for i in range(N_BITS)], 'expected': PAYLOAD,
                        'extracted': bits_v, 'score': score_v, 'match': bits_v == PAYLOAD})
display(df_bits)

wrong = np.flatnonzero(bits_v != PAYLOAD)
if len(wrong) == 0:
    print(f'Semua {N_BITS} bit terbaca benar pada level video. Skor |korelasi| terkecil = {np.abs(score_v).min():.3f} '
          f'(rata-rata {np.abs(score_v).mean():.3f}); STRENGTH={STRENGTH}.')
else:
    ok_abs = np.abs(score_v[bits_v == PAYLOAD]).mean()
    print(f'{len(wrong)} bit salah: indeks {wrong.tolist()}. |skor| rata-rata bit salah = {np.abs(score_v[wrong]).mean():.3f} '
          f'vs bit benar = {ok_abs:.3f}.')
    print(f'Piksel terkena clipping pada blok tsb (frame {AUDIT_FRAME_IDX}): {df_blk.set_index("block").loc[wrong, "clipped_px"].tolist()}.')
    print('Penjelasan (berdasarkan implementasi): ekstraksi hanya mengambil korelasi blok dengan pola ±STRENGTH; skor mendekati 0 berarti energi pola di blok itu '
          'dilemahkan codec (kompresi menghaluskan detail) atau tertutup konten frame yang berkorelasi dengan pola; clipping 0/255 pada area gelap/terang '
          'mengurangi delta efektif.')

## Imperceptibility dan strength sweep

Penilaian memakai **frame asli apa adanya** (tanpa pembesaran), **PSNR** dan **SSIM** asli vs ber-watermark. Untuk modulasi ±`s` murni, MSE = `s²`,
sehingga PSNR ≈ `20·log10(255/s)` (rumus, bukan hasil ukur); nilai terukur dibandingkan dengan rumus ini.

Sweep menjalankan **jalur yang sama** (embed -> `codec_roundtrip` -> ekstraksi -> `evaluate`) untuk beberapa `STRENGTH` pada subset video tersebar merata.
Rekomendasi = `STRENGTH` **terkecil** yang membuat **semua** video sweep tetap `DETECTED`. Hasilnya **tidak diterapkan otomatis**: ubah `STRENGTH` di cell konfigurasi
lalu jalankan ulang dari awal (run_id baru) untuk membuktikan pada seluruh dataset.

In [ ]:
SWEEP_NAMES = VIDEO_NAMES[::max(1, len(VIDEO_NAMES) // SWEEP_MAX_VIDEOS)][:SWEEP_MAX_VIDEOS]
ckpt_sweep = Checkpoint('sweep').load()
sweep_rows = []
for s in SWEEP_STRENGTHS:
    key = f'strength_{s}'
    done = ckpt_sweep.state['done'].get(key)
    if done:
        verify_entry(done)
        part = json.loads(Path(done['artifacts']['result']['path']).read_text(encoding='utf-8'))
        if part['videos'] != SWEEP_NAMES:
            raise CheckpointError(f'daftar video sweep {key} berbeda dari checkpoint')
        rows_s = part['rows']
    else:
        rows_s = []
        for name in tqdm(SWEEP_NAMES, desc=f'Sweep STRENGTH={s}'):
            orig = load_npy(verify_entry(ckpt_frames.state['done'][name]), 'frames')
            wm_s = embed_pixel_watermark(orig, strength=s)
            rec_s, meta_s = codec_roundtrip(wm_s)                            # jalur codec yang sama dengan pipeline utama
            r = evaluate(extract_pixel_watermark(rec_s))
            rows_s.append({'strength': s, 'video': name, 'ber': r['ber'], 'bit_accuracy': r['bit_accuracy'],
                           'detection_status': r['detection_status'], 'detection_p_value': r['detection_p_value'],
                           'psnr_watermark_db': psnr(orig, wm_s), 'ssim_watermark': ssim(orig, wm_s),
                           'psnr_codec_db': psnr(wm_s, rec_s), 'bpp': meta_s['bpp']})
        art = DIRS['reports'] / f'strength_sweep_s{s}.json'
        atomic_write_json(art, {'videos': SWEEP_NAMES, 'rows': rows_s})
        done = {'artifacts': {'result': make_artifact(art)}}
        verify_entry(done)
        ckpt_sweep.commit(stage='EVALUATED', status='IN_PROGRESS', video=key, entry=done)
    sweep_rows += rows_s
ckpt_sweep.commit(stage='EVALUATED', status='DONE')

df_sweep = pd.DataFrame(sweep_rows)
SWEEP_CSV = OUT_ROOT / f'strength_sweep_{RUN_ID}.csv'
atomic_to_csv(df_sweep, SWEEP_CSV)
agg = df_sweep.groupby('strength').agg(
    videos=('video', 'count'), psnr_watermark_db=('psnr_watermark_db', 'mean'), ssim_watermark=('ssim_watermark', 'mean'),
    mean_ber=('ber', 'mean'), worst_ber=('ber', 'max'), mean_bit_accuracy=('bit_accuracy', 'mean'),
    detected=('detection_status', lambda x: int((x == 'DETECTED').sum())), psnr_codec_db=('psnr_codec_db', 'mean'),
    bpp=('bpp', 'mean')).reset_index()
agg['psnr_formula_db'] = 20 * np.log10(255.0 / agg.strength)          # rumus teoritis ±s murni (tanpa clipping), pembanding
agg['detected_all'] = agg.detected == agg.videos
display(agg.style.format({'psnr_watermark_db': '{:.2f}', 'ssim_watermark': '{:.4f}', 'mean_ber': '{:.4f}', 'worst_ber': '{:.4f}',
                          'mean_bit_accuracy': '{:.4f}', 'psnr_codec_db': '{:.2f}', 'bpp': '{:.3f}', 'psnr_formula_db': '{:.2f}'}))
ok_s = agg[agg.detected_all]
if len(ok_s):
    STRENGTH_RECOMMENDED = int(ok_s.strength.min())
    r_ = agg[agg.strength == STRENGTH_RECOMMENDED].iloc[0]
    print(f'Strength minimum yang masih 100% terdeteksi pada {len(SWEEP_NAMES)} video sweep: {STRENGTH_RECOMMENDED} '
          f'(PSNR watermark {r_.psnr_watermark_db:.2f} dB, SSIM {r_.ssim_watermark:.4f}, BER rata-rata {r_.mean_ber:.4f}, BER terburuk {r_.worst_ber:.4f}).')
    print(f'STRENGTH saat ini = {STRENGTH}. Rekomendasi TIDAK diterapkan otomatis; ubah STRENGTH di konfigurasi lalu jalankan ulang dari awal.')
    if r_.worst_ber > 0.5 * (1 - MIN_MATCH / N_BITS):
        print(f'Catatan: BER terburuk {r_.worst_ber:.4f} sudah mendekati batas deteksi {1 - MIN_MATCH / N_BITS:.4f}; margin tipis, pertimbangkan satu level di atasnya.')
else:
    STRENGTH_RECOMMENDED = None
    print('Tidak ada strength pada sweep yang membuat semua video terdeteksi; tidak ada rekomendasi.')

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(agg.strength, agg.psnr_watermark_db, 'o-', label='PSNR terukur (dB)')
ax[0].plot(agg.strength, agg.psnr_formula_db, 'k--', label='rumus 20log10(255/s)')
ax[0].set_xlabel('STRENGTH'); ax[0].set_ylabel('PSNR watermark (dB)'); ax[0].legend(loc='upper right')
a2 = ax[0].twinx(); a2.plot(agg.strength, agg.ssim_watermark, 's-', c='tab:green'); a2.set_ylabel('SSIM watermark', color='tab:green')
ax[0].set_title('Imperceptibility (asli vs ber-watermark)')
ax[1].plot(agg.strength, agg.mean_ber, 'o-', label='BER rata-rata'); ax[1].plot(agg.strength, agg.worst_ber, 'x--', label='BER terburuk')
ax[1].axhline(1 - MIN_MATCH / N_BITS, c='r', ls=':', label=f'batas deteksi (BER <= {1 - MIN_MATCH / N_BITS:.3f})')
ax[1].set_xlabel('STRENGTH'); ax[1].set_ylabel('BER setelah Neural Codec'); ax[1].legend(); ax[1].set_title('Ketahanan terhadap Neural Codec')
plt.tight_layout(); plt.show()

# frame ASLI vs frame ber-watermark pada tiap STRENGTH (tidak diperbesar / tidak dimanipulasi) - dinilai langsung oleh mata
fi = AUDIT_FRAME_IDX
panels = [('Asli', orig_v[fi])] + [(f'STRENGTH={s}', embed_pixel_watermark(orig_v[:fi + 1], strength=s)[fi]) for s in SWEEP_STRENGTHS]
ncol = 4; nrow = int(np.ceil(len(panels) / ncol))
fig, ax = plt.subplots(nrow, ncol, figsize=(4.6 * ncol, 3.6 * nrow))
for a in np.atleast_1d(ax).ravel():
    a.axis('off')
for a, (t, im) in zip(np.atleast_1d(ax).ravel(), panels):
    a.imshow(im, interpolation='nearest'); a.set_title(t)
plt.tight_layout()
fig.savefig(DIRS['reports'] / f'strength_visual_{Path(audit_name).stem}_f{fi}.png', dpi=110)
plt.show()

In [ ]:
required = [METRICS_CSV, METRICS_FP_CSV, MANIFEST_JSON, SWEEP_CSV, CODEC_PROOF_JSON, MAPPING_CSV, PIXEL_AUDIT_CSV,
            *[CKPT_DIR / f'{n}.json' for n in ['run', 'frames', 'embed', 'codec', 'false_positive', 'sweep']]]
missing = [str(p) for p in required if not p.is_file() or p.stat().st_size == 0]
assert not missing, f'artifact wajib belum lengkap: {missing}'
assert set(ckpt_sweep.state['done']) == {f'strength_{s}' for s in SWEEP_STRENGTHS}, 'strength sweep belum lengkap'
for ck in (ckpt_frames, ckpt_embed, ckpt_codec, ckpt_fp):
    assert set(ck.state['done']) == set(VIDEO_NAMES), f'checkpoint {ck.name} belum mencakup semua video'
    for e in ck.state['done'].values():
        verify_entry(e)
assert len(rows) == len(VIDEO_NAMES) and len(rows_fp) == len(VIDEO_NAMES), 'evaluasi / false-positive belum lengkap'
assert ckpt_run.state['pipeline_stage'] in ('EVALUATED', 'COMPLETED'), f'stage harus minimal EVALUATED, sekarang {ckpt_run.state["pipeline_stage"]}'

ckpt_run.commit(stage='COMPLETED', status='COMPLETED')      # hanya setelah metrics + manifest + bukti audit tertulis & terverifikasi
print(f'Run {RUN_ID} COMPLETED ({len(VIDEO_NAMES)} video)')

## Ringkasan visual eksperimen

Empat panel dari data run ini: frame asli, frame ber-watermark (apa adanya), hasil nyata `codec.decompress()`, dan peta deteksi/payload. Tidak ada pembesaran atau manipulasi tampilan.

In [ ]:
fi = AUDIT_FRAME_IDX
orig_v, wm_v, rec_v = load_triplet(audit_name)
row = df[df.video == audit_name].iloc[0]                                    # metrik run utama untuk video ini
H_v, W_v = orig_v.shape[1:3]
bits_vid, score_vid = extract_pixel_watermark(rec_v, return_score=True)     # dari frame hasil rekonstruksi, video-level
res_vid = evaluate(bits_vid)
assert res_vid['bits_correct'] == row['bits_correct'], 'ringkasan tidak konsisten dengan tabel metrik'
res_fr = evaluate(extract_pixel_watermark(rec_v[fi:fi + 1], frame_offset=fi))
psnr_wm_f, ssim_wm_f = psnr(orig_v[fi:fi + 1], wm_v[fi:fi + 1]), ssim(orig_v[fi:fi + 1], wm_v[fi:fi + 1])
psnr_cd_f = psnr(wm_v[fi:fi + 1], rec_v[fi:fi + 1])
verdict = 'PASS' if res_vid['detection_status'] == 'DETECTED' else 'FAIL'
verdict_f = 'PASS' if res_fr['detection_status'] == 'DETECTED' else 'FAIL'

fig, ax = plt.subplots(2, 2, figsize=(15, 12.5), constrained_layout=True)
ax[0, 0].imshow(orig_v[fi], interpolation='nearest'); ax[0, 0].set_title('ORIGINAL FRAME', fontsize=13, weight='bold')
ax[0, 0].set_xlabel(f'{audit_name} | frame {fi}')
ax[0, 1].imshow(wm_v[fi], interpolation='nearest'); ax[0, 1].set_title('WATERMARKED FRAME', fontsize=13, weight='bold')
ax[0, 1].set_xlabel(f'Pixel-domain embedding | STRENGTH={STRENGTH}, KEY={KEY}\nPSNR vs asli: {psnr_wm_f:.2f} dB | SSIM: {ssim_wm_f:.4f}   (tidak diperbesar)')
ax[1, 0].imshow(rec_v[fi], interpolation='nearest'); ax[1, 0].set_title('NEURAL CODEC RECONSTRUCTED FRAME (hasil decompress)', fontsize=13, weight='bold')
ax[1, 0].set_xlabel(f'Codec: {CODEC_NAME} | Quality: {CODEC_QUALITY}\nCompressed bytes: {int(row["bpp"] * len(orig_v) * H_v * W_v / 8)} '
                    f'({len(orig_v)} frame) | BPP: {row["bpp"]:.3f}\nPSNR Codec: frame {psnr_cd_f:.2f} dB | video {row["psnr_codec_db"]:.2f} dB')
ax[1, 1].imshow(rec_v[fi], interpolation='nearest')
boxes_f = block_boxes(H_v, W_v)
good, bad = np.flatnonzero(bits_vid == PAYLOAD), np.flatnonzero(bits_vid != PAYLOAD)
for idx_, ec in ((good, 'lime'), (bad, 'red')):
    _draw_grid(ax[1, 1], [boxes_f[i] for i in idx_], [f'#{i}\n{int(PAYLOAD[i])}|{int(bits_vid[i])}' for i in idx_], ec=ec, fs=7)
ax[1, 1].set_title('WATERMARK DETECTION / PAYLOAD MAPPING', fontsize=13, weight='bold')
ax[1, 1].set_xlabel(f'6 x 8 grid | label: #bit  bit_asli|bit_terbaca | hijau = cocok, merah = salah\n'
                    f'Expected : {PAYLOAD_TEXT}\nExtracted: {res_vid["text_read"]}\n'
                    f'BER      : {res_vid["ber"]:.4f} | Accuracy: {res_vid["bit_accuracy"]:.4f} | Detection: {verdict} (video-level, {len(rec_v)} frame)\n'
                    f'Frame {fi} saja: BER {res_fr["ber"]:.4f} | Detection: {verdict_f}')
for a in ax.ravel():
    a.set_xticks([]); a.set_yticks([])
fig.suptitle(f'Run {RUN_ID}', fontsize=11)
fig.savefig(DIRS['reports'] / f'final_summary_{RUN_ID}.png', dpi=110)
plt.show()